# Preprocessing
what all we will do

- Combine ticket subject + ticket description into one column to have both subject for clean topic of ticket and description will add context to it. Model will require only one column then

- lowercase emntire text

- strip extra white spaces for better tockenization

- encode ticket type like "technical issue" into 0 and "biilong inquiry" to 1 as the model works with numbers, not strings

- Encode ticket priority to numbers the same way as ticket tye

- save the clean dataframe

In [170]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import re  ##regex to remobe the place holders

In [171]:
df = pd.read_csv('../Dataset/sample_tickets_dataset.csv')
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Sara Mensah,saramensah@outlook.com,33,Male,Massage Gun Pro,2023-03-30,Technical Issue,Something went wrong after the latest update,I've had my Massage Gun Pro for a while now bu...,Resolved,Replacement unit dispatched after fault confir...,Medium,Social Media,1d 2h,2h 45m,3.0
1,2,Derek Khan,derek.khan@me.com,38,Male,LaptopX 15,2023-10-03,Product Inquiry,I have a few questions about a product,Before I place an order for the LaptopX 15 I'd...,In Progress,Pending,Medium,Social Media,8h,Pending,NaN
2,3,Felix Brown,felixbrown@gmail.com,25,Male,SoundWave Earbuds,2024-04-30,Technical Issue,Something went wrong after the latest update,I purchased the SoundWave Earbuds 4 weeks ago ...,Open,Pending,High,Social Media,4h,Pending,NaN
3,4,Andre Wilson,andre.wilson@live.com,25,Female,KitchenBlast Blender,2023-05-28,Product Inquiry,Product question from a potential customer,i'm very close to ordering teh KitchenBlast Bl...,Resolved,Customer directed to detailed setup guide.,Medium,Chat,12h,4d 6h,4.0
4,5,Ethan Goldstein,ethan.goldstein@protonmail.com,64,Male,Massage Gun Pro,2024-03-30,Refund Request,Help with processing a refund,I'll keep this brief — two problems. I've noti...,Resolved,Partial refund approved per return policy.,Low,Chat,2h 30m,4d 6h,5.0


In [172]:
## removing unnessary columns
df = df.drop(columns=['Ticket ID','Customer Name', 'Customer Email', 'Customer Age', 'Customer Gender',
               'Date of Purchase','First Response Time','Time to Resolution','Resolution', 
               "Customer Satisfaction Rating"])

In [173]:
print(df.shape)
print(df.dtypes)


(4500, 7)
Product Purchased     str
Ticket Type           str
Ticket Subject        str
Ticket Description    str
Ticket Status         str
Ticket Priority       str
Ticket Channel        str
dtype: object


In [174]:
## combining subject and description
df['combine_text'] = df["Ticket Subject"] + " " + df["Ticket Description"]

df['combine_text'].head()

0    Something went wrong after the latest update I...
1    I have a few questions about a product Before ...
2    Something went wrong after the latest update I...
3    Product question from a potential customer i'm...
4    Help with processing a refund I'll keep this b...
Name: combine_text, dtype: str

In [175]:
## changing all the text to lower and removing unneccessary spaces
df['combine_text'] = df['combine_text'].str.lower()

In [176]:
df['combine_text'] = df['combine_text'].str.strip()

In [177]:
df['combine_text'].head()

0    something went wrong after the latest update i...
1    i have a few questions about a product before ...
2    something went wrong after the latest update i...
3    product question from a potential customer i'm...
4    help with processing a refund i'll keep this b...
Name: combine_text, dtype: str

In [178]:
## Label encoding ticket type and ticket prioirty

# encoding ticket type

le_type = LabelEncoder()
df['ticket_type_encoded'] = le_type.fit_transform(df['Ticket Type'])

print(dict(zip(le_type.classes_, le_type.transform(le_type.classes_))))

{'Account Access': np.int64(0), 'Billing Inquiry': np.int64(1), 'Product Inquiry': np.int64(2), 'Refund Request': np.int64(3), 'Technical Issue': np.int64(4)}


In [179]:
## encoding ticket priority

le_priority = LabelEncoder()
df['ticket_priority_encoded'] = le_priority.fit_transform(df['Ticket Priority'])

print(dict(zip(le_priority.classes_, le_priority.transform(le_priority.classes_))))

{'Critical': np.int64(0), 'High': np.int64(1), 'Low': np.int64(2), 'Medium': np.int64(3)}


In [180]:
print(df[['Ticket Type', 'ticket_type_encoded', 'Ticket Priority', 'ticket_priority_encoded']].head(10))

       Ticket Type  ...  ticket_priority_encoded
0  Technical Issue  ...                        3
1  Product Inquiry  ...                        3
2  Technical Issue  ...                        1
3  Product Inquiry  ...                        3
4   Refund Request  ...                        2
5  Billing Inquiry  ...                        3
6  Product Inquiry  ...                        1
7  Billing Inquiry  ...                        0
8   Account Access  ...                        2
9  Product Inquiry  ...                        0

[10 rows x 4 columns]


In [181]:
## saving the encoders

import joblib
joblib.dump(le_type , '../models/le_type.pkl')
joblib.dump(le_priority, "../models/le_priority.pkl")

print("encoders saved")

encoders saved


In [182]:
print("Class distribution:")
print(df['ticket_type_encoded'].value_counts())

Class distribution:
ticket_type_encoded
4    1072
2     887
0     874
3     867
1     800
Name: count, dtype: int64


In [183]:
df.head()

,Product Purchased,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Ticket Priority,Ticket Channel,combine_text,ticket_type_encoded,ticket_priority_encoded
0,Massage Gun Pro,Technical Issue,Something went wrong after the latest update,I've had my Massage Gun Pro for a while now bu...,Resolved,Medium,Social Media,something went wrong after the latest update i...,4,3
1,LaptopX 15,Product Inquiry,I have a few questions about a product,Before I place an order for the LaptopX 15 I'd...,In Progress,Medium,Social Media,i have a few questions about a product before ...,2,3
2,SoundWave Earbuds,Technical Issue,Something went wrong after the latest update,I purchased the SoundWave Earbuds 4 weeks ago ...,Open,High,Social Media,something went wrong after the latest update i...,4,1
3,KitchenBlast Blender,Product Inquiry,Product question from a potential customer,i'm very close to ordering teh KitchenBlast Bl...,Resolved,Medium,Chat,product question from a potential customer i'm...,2,3
4,Massage Gun Pro,Refund Request,Help with processing a refund,I'll keep this brief — two problems. I've noti...,Resolved,Low,Chat,help with processing a refund i'll keep this b...,3,2


In [184]:
## dropping the columns we do not require any more

df = df.drop(columns=['Ticket Description'])
df.head(3)

,Product Purchased,Ticket Type,Ticket Subject,Ticket Status,Ticket Priority,Ticket Channel,combine_text,ticket_type_encoded,ticket_priority_encoded
0,Massage Gun Pro,Technical Issue,Something went wrong after the latest update,Resolved,Medium,Social Media,something went wrong after the latest update i...,4,3
1,LaptopX 15,Product Inquiry,I have a few questions about a product,In Progress,Medium,Social Media,i have a few questions about a product before ...,2,3
2,SoundWave Earbuds,Technical Issue,Something went wrong after the latest update,Open,High,Social Media,something went wrong after the latest update i...,4,1


In [185]:
## we kept the rest of the columns for following reasons:

# combine_text — model input
# ticket_type_encoded — model target 1
# ticket_priority_encoded — model target 2
# Ticket Type — for display in Streamlit
# Ticket Priority — for display in Streamlit
# Product Purchased — useful context to display
# Ticket Subject — useful for display in results

In [186]:
## saving the new dataset

df.to_csv('../Dataset/cleaned_tickets.csv',index=False)
print("dataset saved. Shape:", df.shape)
print("columns: ", df.columns.tolist())

dataset saved. Shape: (4500, 9)
columns:  ['Product Purchased', 'Ticket Type', 'Ticket Subject', 'Ticket Status', 'Ticket Priority', 'Ticket Channel', 'combine_text', 'ticket_type_encoded', 'ticket_priority_encoded']
